# Impacto del conflicto Rusia-Ucrania en los precios de los alimentos en Europa (2020-2025)

## Fase 2: Obtención, limpieza y transformación de datos

Este notebook corresponde a la Fase 2 del proyecto y tiene como objetivo implementar un pipeline reproducible para la obtención, exploración, limpieza, transformación y validación del conjunto de datos Global Food Prices Database (WFP).

Durante esta etapa se trabajará con información correspondiente al período 2020-2025, enfocándose en países europeos y utilizando precios expresados en dólares estadounidenses (USD). Las actividades desarrolladas permitirán preparar un conjunto de datos consistente, íntegro y apto para las siguientes fases de análisis exploratorio y evaluación del impacto del conflicto Rusia-Ucrania sobre los precios de los alimentos.

Todas las transformaciones realizadas serán documentadas y justificadas técnicamente, manteniendo trazabilidad entre los datos originales, las decisiones metodológicas adoptadas y el dataset final resultante.


# Contexto

El conflicto entre Rusia y Ucrania, iniciado en febrero de 2022, generó importantes alteraciones en los mercados internacionales debido a interrupciones en las cadenas de suministro, restricciones comerciales y fluctuaciones en los costos de transporte y energía.

Debido a que ambos países poseen una participación relevante en la producción y exportación de diversos productos agrícolas, resulta pertinente analizar si es posible identificar variaciones en los precios de los alimentos en países europeos durante el período comprendido entre 2020 y 2025.

Para abordar esta problemática se utilizará el conjunto de datos *Global Food Prices Database* del World Food Programme (WFP), el cual contiene información histórica de precios de alimentos para distintos países, mercados y categorías de productos.

En esta fase se desarrollarán las actividades de obtención, exploración, limpieza, transformación y validación de los datos, con el objetivo de construir un conjunto de datos consistente y preparado para las etapas posteriores de análisis exploratorio e interpretación de resultados.


# Problemática

¿Cómo ha impactado el conflicto Rusia-Ucrania en la evolución de los precios de los alimentos en Europa durante el período 2020-2025?

Se busca identificar tendencias temporales, posibles incrementos de precios y diferencias entre países europeos utilizando información histórica proporcionada por el World Food Programme (WFP).

Para responder esta pregunta será necesario preparar un conjunto de datos confiable mediante procesos de obtención, limpieza, transformación y validación, asegurando la calidad e integridad de la información que será utilizada en las etapas posteriores de análisis exploratorio y evaluación de resultados.


# Objetivo General

Analizar la evolución de los precios de los alimentos en Europa entre 2020 y 2025 para identificar posibles efectos asociados al conflicto Rusia-Ucrania, utilizando un conjunto de datos preparado mediante procesos de limpieza, transformación y validación que garanticen su calidad y consistencia para las etapas posteriores de análisis.


# Objetivos Específicos

* Consolidar y explorar los datos correspondientes al período 2020-2025 provenientes de la base Global Food Prices Database (WFP).
* Identificar y tratar valores faltantes, registros inconsistentes y posibles duplicados presentes en el conjunto de datos.
* Aplicar transformaciones necesarias sobre las variables para facilitar su análisis posterior.
* Filtrar y preparar la información correspondiente a países europeos incluidos en el alcance del proyecto.
* Validar la integridad, consistencia y calidad del dataset resultante mediante verificaciones técnicas y controles de calidad.
* Generar un conjunto de datos limpio y reproducible que sirva como base para las siguientes fases de análisis exploratorio y evaluación de resultados.


# Alcance del Proyecto

## Incluye

* Datos del World Food Programme (WFP).
* Países europeos.
* Período comprendido entre 2020 y 2025.
* Obtención, exploración, limpieza y transformación de datos.
* Validación y preparación de un dataset reproducible para análisis posteriores.

## No incluye

* Variables económicas externas no presentes en el dataset.
* Datos correspondientes a regiones fuera de Europa.
* Establecimiento de relaciones causales directas entre el conflicto Rusia-Ucrania y los cambios observados en los precios.
* Modelos predictivos o técnicas avanzadas de aprendizaje automático en esta fase del proyecto.


# Carga automatizada de archivos CSV

Con el objetivo de reducir errores asociados a la carga manual de datos y mejorar la reproducibilidad del proceso, se utiliza la biblioteca `pathlib` para identificar automáticamente todos los archivos CSV disponibles en el directorio de trabajo.

Esta estrategia permite incorporar nuevos archivos al proyecto sin necesidad de modificar el código de carga, facilitando el mantenimiento y la escalabilidad del pipeline de procesamiento de datos.

Adicionalmente, se implementa una función encargada de leer y consolidar los distintos archivos en un único DataFrame. Esta aproximación permite trabajar de manera uniforme con la información correspondiente al período 2020-2025, manteniendo la trazabilidad de los registros mediante la incorporación de la columna `archivo_origen`, la cual identifica el archivo desde donde fue obtenido cada dato.

La utilización de funciones específicas para la carga de información favorece la reutilización del código, mejora la organización del notebook y contribuye a la construcción de un flujo de trabajo reproducible y verificable.


# Estructura del dataset

El conjunto de datos consolidado contiene información histórica de precios de alimentos recopilada por el World Food Programme (WFP) para el período 2020-2025. Las principales variables utilizadas dentro del proyecto corresponden a:

* `countryiso3`: código del país.
* `date`: fecha de registro.
* `market`: mercado o localidad donde se registró el precio.
* `category`: categoría del producto.
* `commodity`: nombre del producto alimentario.
* `currency`: moneda original del registro.
* `price`: precio reportado en moneda local.
* `usdprice`: precio convertido a dólares estadounidenses (USD).
* `year`: año derivado de la fecha de registro.
* `month`: mes derivado de la fecha de registro.

Estas variables constituyen la base para los procesos de limpieza, validación, transformación y análisis exploratorio que serán desarrollados durante las siguientes fases del proyecto. La utilización de precios expresados en dólares estadounidenses permite realizar comparaciones homogéneas entre países y períodos de tiempo.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Ruta base donde se encuentran los archivos CSV.
DATA_DIR = Path("../data")

# Busca automáticamente todos los archivos CSV dentro de la carpeta data.
archivos_csv = sorted(DATA_DIR.glob("*.csv"))

print(f"Archivos CSV encontrados: {len(archivos_csv)}")

Archivos CSV encontrados: 7


In [2]:

def cargar_archivos_csv(rutas):
    """
    Carga múltiples archivos CSV y los combina en un solo DataFrame.

    Parámetros:
    rutas: list
        Lista de rutas de archivos CSV.

    Retorna:
    pandas.DataFrame
        DataFrame consolidado con todos los archivos cargados.
    """
    dataframes = []

    for ruta in rutas:
        df_temp = pd.read_csv(ruta)
        df_temp["archivo_origen"] = ruta.name
        dataframes.append(df_temp)

    df_consolidado = pd.concat(dataframes, ignore_index=True)

    return df_consolidado

def resumen_dataset(df):
    """
    Genera un resumen inicial del conjunto de datos.
    """
    resumen = {
        "filas": df.shape[0],
        "columnas": df.shape[1],
        "duplicados": df.duplicated().sum(),
        "columnas_disponibles": list(df.columns)
    }

    return resumen

def revisar_valores_nulos(df):
    """
    Calcula la cantidad y porcentaje de valores nulos por columna.
    """
    valores_nulos = df.isnull().sum()
    porcentaje_nulos = (valores_nulos / len(df)) * 100

    resumen_nulos = pd.DataFrame({
        "valores_nulos": valores_nulos,
        "porcentaje_nulos": porcentaje_nulos.round(2)
    })

    return resumen_nulos.sort_values(by="porcentaje_nulos", ascending=False)

def revisar_tipos_datos(df):
    """
    Retorna los tipos de datos de cada columna del DataFrame.
    """
    return pd.DataFrame({
        "columna": df.columns,
        "tipo_dato": df.dtypes.astype(str).values
    })

def convertir_fecha(df):
    """
    Convierte la columna date al tipo datetime.
    """
    df = df.copy()

    df["date"] = pd.to_datetime(df["date"])

    return df

def crear_variables_temporales(df):
    """
    Genera variables temporales derivadas de la fecha.
    """
    df = df.copy()

    df["year"] = df["date"].dt.year
    df["month"] = df["date"].dt.month

    return df

def filtrar_paises_europeos(df):
    """
    Filtra los países europeos disponibles
    dentro del dataset utilizado en el proyecto.
    """
    paises_europeos = [
        "UKR",
        "TUR",
        "MDA",
        "BLR",
        "RUS"
    ]

    return df[df["countryiso3"].isin(paises_europeos)].copy()


In [3]:
df = cargar_archivos_csv(archivos_csv)
df.head()

,countryiso3,date,admin1,admin2,market,market_id,latitude,longitude,category,commodity,commodity_id,unit,priceflag,pricetype,currency,price,usdprice,archivo_origen,year,month
0,BLR,2020-01-15,Minsk City,Minsk City,Minsk,2618,53.91,27.56,cereals and tubers,Bread (high grade flour),459,KG,actual,Retail,BYR,2.89,1.36,wfp_europe_2020_2025_clean.csv,2020.0,1.0
1,BLR,2020-01-15,Minsk City,Minsk City,Minsk,2618,53.91,27.56,cereals and tubers,Potatoes,83,KG,actual,Retail,BYR,0.84,0.40,wfp_europe_2020_2025_clean.csv,2020.0,1.0
2,BLR,2020-01-15,Minsk City,Minsk City,Minsk,2618,53.91,27.56,cereals and tubers,Wheat flour,58,KG,actual,Retail,BYR,1.28,0.60,wfp_europe_2020_2025_clean.csv,2020.0,1.0
3,BLR,2020-02-15,Minsk City,Minsk City,Minsk,2618,53.91,27.56,cereals and tubers,Bread (high grade flour),459,KG,actual,Retail,BYR,2.88,1.31,wfp_europe_2020_2025_clean.csv,2020.0,2.0
4,BLR,2020-02-15,Minsk City,Minsk City,Minsk,2618,53.91,27.56,cereals and tubers,Potatoes,83,KG,actual,Retail,BYR,0.83,0.38,wfp_europe_2020_2025_clean.csv,2020.0,2.0


In [4]:
pd.DataFrame([resumen_dataset(df)])

,filas,columnas,duplicados,columnas_disponibles
0,2822001,20,0,"[countryiso3, date, admin1, admin2, market, ma..."


In [5]:
revisar_valores_nulos(df)

,valores_nulos,porcentaje_nulos
year,2748895,97.41
month,2748895,97.41
admin2,4678,0.17
admin1,4678,0.17
longitude,4678,0.17
latitude,4678,0.17
usdprice,4375,0.16
date,0,0.00
countryiso3,0,0.00
category,0,0.00


In [6]:
df = convertir_fecha(df)
print(df["date"].dtype)

datetime64[us]


In [7]:
df = crear_variables_temporales(df)
df[["date", "year", "month"]].head()

,date,year,month
0,2020-01-15,2020,1
1,2020-01-15,2020,1
2,2020-01-15,2020,1
3,2020-02-15,2020,2
4,2020-02-15,2020,2


In [8]:
print("Años disponibles:")
print(sorted(df["year"].unique()))

print("\nMeses disponibles:")
print(sorted(df["month"].unique()))

Años disponibles:
[np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025)]

Meses disponibles:
[np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6), np.int32(7), np.int32(8), np.int32(9), np.int32(10), np.int32(11), np.int32(12)]


In [9]:
df_europa = filtrar_paises_europeos(df)

print("Registros originales:", len(df))
print("Registros Europa:", len(df_europa))

print("\nPaíses incluidos:")
print(df_europa["countryiso3"].value_counts())

Registros originales: 2822001
Registros Europa: 146612

Países incluidos:
countryiso3
UKR    131452
TUR     12628
MDA      2322
BLR       174
RUS        36
Name: count, dtype: int64


El dataset no contiene información para la totalidad de los países europeos. Por esta razón, el análisis se realizará exclusivamente sobre los países disponibles dentro de la cobertura del WFP.

In [10]:
nulos_europa = df_europa.isnull().sum().sort_values(ascending=False)

print(nulos_europa)

admin2            2280
admin1            2280
latitude          2280
longitude         2280
usdprice           400
date                 0
countryiso3          0
market_id            0
category             0
commodity            0
commodity_id         0
market               0
unit                 0
priceflag            0
currency             0
pricetype            0
price                0
archivo_origen       0
year                 0
month                0
dtype: int64


In [11]:
porcentaje_nulos = (
    df_europa.isnull().sum() / len(df_europa) * 100
).round(2).sort_values(ascending=False)

print(porcentaje_nulos)

admin2            1.56
admin1            1.56
latitude          1.56
longitude         1.56
usdprice          0.27
date              0.00
countryiso3       0.00
market_id         0.00
category          0.00
commodity         0.00
commodity_id      0.00
market            0.00
unit              0.00
priceflag         0.00
currency          0.00
pricetype         0.00
price             0.00
archivo_origen    0.00
year              0.00
month             0.00
dtype: float64


In [12]:
df_europa[df_europa["usdprice"].isnull()][
    ["countryiso3", "commodity", "price", "currency", "usdprice"]
].head(20)

,countryiso3,commodity,price,currency,usdprice
2609516,MDA,Bread,8.5,PRB,NaN
2609517,MDA,Buckwheat,22.6,PRB,NaN
2609518,MDA,Maize flour,15.1,PRB,NaN
2609519,MDA,Oat flakes,21.3,PRB,NaN
2609520,MDA,Pasta (macaroni),23.0,PRB,NaN
2609521,MDA,Potatoes,12.9,PRB,NaN
2609522,MDA,"Rice (medium grain, imported)",27.2,PRB,NaN
2609523,MDA,Wheat flour (high quality),21.5,PRB,NaN
2609524,MDA,"Meat (chicken, legs)",46.7,PRB,NaN
2609525,MDA,Meat (pork),100.0,PRB,NaN


In [13]:
df_europa[df_europa["usdprice"].isnull()]["countryiso3"].value_counts()

countryiso3
MDA    400
Name: count, dtype: int64

In [14]:
print("Registros antes:", len(df_europa))

df_europa = df_europa.dropna(subset=["usdprice"])

print("Registros después:", len(df_europa))
print("Nulos usdprice:", df_europa["usdprice"].isnull().sum())

Registros antes: 146612
Registros después: 146212
Nulos usdprice: 0


In [15]:
print("Duplicados:", df_europa.duplicated().sum())

Duplicados: 0


In [16]:
def validar_dataset(df):
    """
    Genera un resumen de validación del dataset.
    """

    print("Filas:", len(df))
    print("Columnas:", len(df.columns))
    print("Duplicados:", df.duplicated().sum())
    print("Nulos totales:", df.isnull().sum().sum())

    return

In [17]:
validar_dataset(df_europa)

Filas: 146212
Columnas: 20
Duplicados: 0
Nulos totales: 9120


In [18]:
print("Año mínimo:", df_europa["year"].min())
print("Año máximo:", df_europa["year"].max())

Año mínimo: 2020
Año máximo: 2025


In [19]:
(df_europa["usdprice"] < 0).sum()

np.int64(0)

# Exportación del dataset procesado

Una vez finalizadas las etapas de limpieza, transformación y validación, se genera una versión procesada del conjunto de datos. Este archivo será utilizado como fuente de información para las siguientes fases del proyecto y permite mantener la trazabilidad entre los datos originales y los resultados obtenidos.

In [20]:
df_europa.to_csv(
    "../data/processed/wfp_europe_2020_2025_clean.csv",
    index=False
)

In [21]:
df_validacion = pd.read_csv(
    "../data/processed/wfp_europe_2020_2025_clean.csv"
)

print(df_validacion.shape)

(146212, 20)


## Justificación técnica del uso de funciones y transformaciones aplicadas

El uso de funciones permitió estructurar el pipeline de procesamiento de datos de forma modular, ordenada y reproducible. Cada función fue diseñada para cumplir una tarea específica dentro del flujo de trabajo, favoreciendo la reutilización del código, la trazabilidad de las operaciones realizadas y la validación independiente de cada etapa del proceso.

En primer lugar, se implementaron funciones para la carga automatizada y consolidación de los archivos CSV correspondientes al período 2020-2025. Posteriormente, se desarrollaron funciones orientadas al diagnóstico inicial de calidad de datos, incluyendo la revisión de valores faltantes, tipos de datos y registros duplicados.

Como parte del preprocesamiento, se aplicaron transformaciones sobre la variable de fecha mediante la conversión al tipo `datetime`, permitiendo la generación de variables temporales derivadas (`year` y `month`) necesarias para futuros análisis cronológicos. Asimismo, se implementó un procedimiento de filtrado geográfico para conservar únicamente los países europeos incluidos dentro del alcance del proyecto.

Durante la etapa de limpieza se identificaron registros con valores faltantes en la variable `usdprice`, considerada fundamental para la comparación de precios entre países. Debido a que dichos registros representaban un porcentaje reducido del conjunto de datos y no existía información suficiente para reconstruirlos de manera confiable, se optó por su eliminación.

Finalmente, se realizaron procesos de validación para verificar la integridad y consistencia del conjunto de datos resultante, comprobando la ausencia de errores críticos, la correcta generación de variables y la disponibilidad de información para todo el período analizado. El resultado de este proceso corresponde a un dataset limpio, validado y preparado para las siguientes fases de análisis exploratorio y evaluación de resultados.

Esta estrategia permite mantener coherencia entre entrada, procesamiento y salida de datos, además de garantizar que cualquier integrante del equipo pueda reproducir completamente el flujo de trabajo utilizando el entorno configurado y documentado en el repositorio GitHub.


# Conclusiones

Durante la Fase 2 se implementó un pipeline reproducible para la obtención, limpieza, transformación y validación del conjunto de datos seleccionado para el proyecto.

Se consolidaron los archivos correspondientes al período 2020-2025 en un único DataFrame, permitiendo trabajar con una estructura homogénea y consistente para las etapas posteriores del análisis. Asimismo, se realizó una evaluación de la calidad de los datos mediante la revisión de tipos de datos, registros duplicados y valores faltantes.

Como parte del proceso de transformación, la variable de fecha fue convertida al formato adecuado para análisis temporal y se generaron las variables derivadas `year` y `month`, las cuales facilitarán el estudio de tendencias a lo largo del tiempo. Además, se aplicó un filtrado geográfico para conservar únicamente los países europeos disponibles dentro del conjunto de datos y alineados con el alcance definido para el proyecto.

Respecto al tratamiento de valores faltantes, se identificaron registros sin información en la variable `usdprice`, considerada fundamental para la comparación de precios entre países. Debido a su baja proporción respecto al volumen total de datos y a la imposibilidad de reconstruirlos de forma confiable, dichos registros fueron eliminados, permitiendo obtener un conjunto de datos más consistente para los análisis posteriores.

Finalmente, las validaciones realizadas permitieron confirmar la integridad y coherencia del dataset resultante, generando una versión limpia y preparada para las siguientes fases del proyecto, orientadas al análisis exploratorio, identificación de tendencias y evaluación de posibles efectos asociados al conflicto Rusia-Ucrania sobre los precios de los alimentos.
